# KalKalori — Simulation Smoke Test

Notebook version of `core/tests/simulation_smoke.py`. Exercises
`BareTubeHeatExchanger.simulate(...)` (v0.5.x mean-property simulation)
WITHOUT CoolProp / IAPWS / PsychroLib: it uses a tiny linear
temperature-dependent provider plus `ConstantPropertyProvider` so the outer
iteration loop, the forced-averaged single-pass short-circuit, convergence
diagnostics, non-convergence handling, and `surface_margin` derating can all
be exercised without external property backends.

For Rating (closing a known heat balance to get overdesign/margin), see
`core/tests/heat_balance_rating_smoke.py`.

In [ ]:
from pathlib import Path
import sys
from dataclasses import dataclass

import pandas as pd

# Make the notebook usable both from repository root and from core/tests.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]

for candidate in candidate_roots:
    if (candidate / "core").is_dir():
        workspace_root = candidate
        break
else:
    raise RuntimeError("Could not find repository root containing the 'core' directory.")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print("Workspace root:", workspace_root)

In [ ]:
from core.geometry.tube import BareTube
from core.geometry.bundle import TubeBundle
from core.properties.common import FluidTransportProperties
from core.properties.fluids import ConstantPropertyProvider
from core.models.bare_tube import BareTubeHeatExchanger
from core.models.simulation import HXSideInput

print("Imports completed.")

## Setup: Toy Provider, Geometry, Helpers

`LinearGasProvider` is not physical: it only needs to make properties move
with temperature so the mean-property iteration produces a visibly different
result from an inlet-only evaluation. Ignores pressure.

In [ ]:
@dataclass(frozen=True)
class LinearGasProvider:
    rho0: float
    mu0: float
    k0: float
    cp0: float
    T_ref: float = 300.0
    rho_slope: float = -1.0 / 300.0   # rho falls with T (ideal-gas-like)
    mu_slope: float = 1.5e-3
    k_slope: float = 1.5e-3
    cp_slope: float = 2.0e-4

    def at(self, T: float, p: float) -> FluidTransportProperties:
        dT = T - self.T_ref
        rho = max(self.rho0 * (1.0 + self.rho_slope * dT), 1e-3)
        mu = max(self.mu0 * (1.0 + self.mu_slope * dT), 1e-7)
        k = max(self.k0 * (1.0 + self.k_slope * dT), 1e-4)
        cp = max(self.cp0 * (1.0 + self.cp_slope * dT), 1.0)
        return FluidTransportProperties(rho=rho, mu=mu, k=k, cp=cp)


def build_bundle() -> TubeBundle:
    """Geometry from the v0.4.5 gas-gas test case."""
    tube = BareTube(
        D_i=25e-3 - 2 * 1.5e-3,
        D_o=25e-3,
        length_total=2.8,
        length_effective=2.8,
        wall_k=50.0,
    )
    return TubeBundle(
        tube=tube,
        n_rows=36,
        n_tubes_per_row=56,
        pitch_transverse=35e-3,
        pitch_longitudinal=35e-3,
        layout="staggered",
        n_passes_tube=2,
        flow_arrangement="counterflow",
    )


def c_to_k(t_c: float) -> float:
    return t_c + 273.15


def kgh_to_kgs(m: float) -> float:
    return m / 3600.0


hx = BareTubeHeatExchanger(build_bundle())

inside_var = HXSideInput(
    provider=LinearGasProvider(rho0=1.13, mu0=1.9e-5, k0=0.027, cp0=1007.0),
    m_dot=kgh_to_kgs(18_220.0), T_in=c_to_k(30.0), p=101_325.0,
)
outside_var = HXSideInput(
    provider=LinearGasProvider(rho0=0.50, mu0=3.1e-5, k0=0.052, cp0=1180.0),
    m_dot=kgh_to_kgs(28_380.0), T_in=c_to_k(400.0), p=101_325.0,
)

print("Geometry and variable-property side inputs ready.")

## Case 1 — Default `simulate()`: Mean-Property Iteration (Variable Properties)

In [ ]:
res = hx.simulate(inside_var, outside_var)

assert res.converged and res.iterations > 1, "case 1 should iterate and converge"

pd.Series({
    "converged": res.converged,
    "iterations": res.iterations,
    "residual_q_rel": res.residual_q_rel,
    "q_kW": res.q / 1e3,
    "UA_W_K": res.UA,
    "U_mean_W_m2K": res.U_mean,
    "T_mean_inside_C": res.T_mean_inside - 273.15,
    "T_mean_outside_C": res.T_mean_outside - 273.15,
    "T_out_inside_C": res.T_out_inside - 273.15,
    "T_out_outside_C": res.T_out_outside - 273.15,
}, name="value").to_frame()

## Case 2 — Forced Averaged Properties (`ConstantPropertyProvider`) → 1 Pass

In [ ]:
inside_c = HXSideInput(
    provider=ConstantPropertyProvider(
        FluidTransportProperties(rho=1.13, mu=1.9e-5, k=0.027, cp=1007.0)
    ),
    m_dot=kgh_to_kgs(18_220.0), T_in=c_to_k(30.0), p=101_325.0,
)
outside_c = HXSideInput(
    provider=ConstantPropertyProvider(
        FluidTransportProperties(rho=0.50, mu=3.1e-5, k=0.052, cp=1180.0)
    ),
    m_dot=kgh_to_kgs(28_380.0), T_in=c_to_k(400.0), p=101_325.0,
)

res2 = hx.simulate(inside_c, outside_c)

assert res2.converged and res2.iterations == 1, "case 2 must be a single pass"

pd.Series({
    "converged": res2.converged,
    "iterations": res2.iterations,
    "q_kW": res2.q / 1e3,
    "T_out_inside_C": res2.T_out_inside - 273.15,
    "T_out_outside_C": res2.T_out_outside - 273.15,
}, name="value").to_frame()

## Case 3 — `iterate=False` Escape Hatch (Variable Properties → Inlet-Only, 1 Pass)

In [ ]:
res_inlet = hx.simulate(inside_var, outside_var, iterate=False)

assert res_inlet.converged and res_inlet.iterations == 1, "iterate=False -> single pass"

dq = (res.q - res_inlet.q) / res_inlet.q * 100.0
print(f"mean-property vs inlet-only duty shift: {dq:+.2f}%")

pd.Series({
    "converged": res_inlet.converged,
    "iterations": res_inlet.iterations,
    "q_kW": res_inlet.q / 1e3,
}, name="value").to_frame()

## Case 4 — Force Non-Convergence (`max_iter=3`) → Warning, No Hang

In [ ]:
res4 = hx.simulate(inside_var, outside_var, max_iter=3, relaxation_factor=0.2)

assert not res4.converged and res4.iterations == 3, "case 4 must not converge"

print(f"converged / iterations : {res4.converged} / {res4.iterations}")
for w in (res4.warnings or []):
    if w.source == "simulation":
        print(f"warning[{w.severity}] {w.code}")

## Case 5 — `surface_margin=0.0` Must Reproduce Case 2 Bit-for-Bit

In [ ]:
res_margin0 = hx.simulate(inside_c, outside_c, surface_margin=0.0)

assert res_margin0.q == res2.q, "surface_margin=0.0 must reproduce q bit-for-bit"
assert res_margin0.T_out_inside == res2.T_out_inside
assert res_margin0.T_out_outside == res2.T_out_outside
assert res_margin0.Q_full == res_margin0.Q_derated == res_margin0.q

pd.Series({
    "q_kW": res_margin0.q / 1e3,
    "Q_full_kW": res_margin0.Q_full / 1e3,
    "Q_derated_kW": res_margin0.Q_derated / 1e3,
    "Q_full == Q_derated": res_margin0.Q_full == res_margin0.Q_derated,
}, name="value").to_frame()

## Case 6 — `surface_margin` > 0 Derates Duty Monotonically

In [ ]:
res_margin_low = hx.simulate(inside_c, outside_c, surface_margin=0.2)
res_margin_high = hx.simulate(inside_c, outside_c, surface_margin=0.5)

assert res_margin_low.q < res_margin0.q, "margin=0.2 must derate duty below margin=0"
assert res_margin_high.q < res_margin_low.q, "margin=0.5 must derate duty further than 0.2"
assert res_margin_high.Q_full > res_margin_high.Q_derated, "Q_full must exceed Q_derated when margin > 0"
assert res_margin_low.Q_full > res_margin_low.Q_derated

pd.DataFrame([
    {"surface_margin": 0.0, "q_kW": res_margin0.q / 1e3},
    {"surface_margin": 0.2, "q_kW": res_margin_low.q / 1e3},
    {"surface_margin": 0.5, "q_kW": res_margin_high.q / 1e3},
])

## Energy-Balance Sanity Check (Case 1)

In [ ]:
q_in = inside_var.m_dot * res.inside_props_mean.cp * (res.T_out_inside - inside_var.T_in)
q_out = outside_var.m_dot * res.outside_props_mean.cp * (outside_var.T_in - res.T_out_outside)

print(f"energy-balance: q_in={q_in/1e3:.2f} kW  q_out={q_out/1e3:.2f} kW  q={res.q/1e3:.2f} kW")

assert abs(q_in - res.q) / res.q < 0.02
assert abs(q_out - res.q) / res.q < 0.02

print("\nALL SMOKE CHECKS PASSED")